In [ ]:
#Loading the data
from langchain_community.document_loaders import PyPDFLoader
document=PyPDFLoader("sample.pdf")
doc=document.load()



In [16]:
from dotenv import load_dotenv
load_dotenv()
import os
os.environ["GOOGLE_API_KEY"]=os.getenv("LANGCHAIN_KEY")


python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 3


In [8]:

#Splitting the documents into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(doc)
documents[:5]


[Document(metadata={'producer': 'GPL Ghostscript 9.52', 'creator': 'LaTeX with hyperref', 'creationdate': "D:20240614052145Z00'00'", 'moddate': "D:20240614052145Z00'00'", 'title': '', 'author': '', 'subject': '', 'keywords': '', 'source': 'sample.pdf', 'total_pages': 27, 'page': 0, 'page_label': '1'}, page_content='Advanced Detection of Sucker Rod Pump Faults\nUsing Computer Vision and Dynamometer Card\nAnalysis\nAmr Gharieb Ali Ramadan\xa0\nApache Egypt JV (KPC)\nAhmed Algarhy\xa0\nMarietta College\nMohamed Adel Gabry\xa0\nApache Egypt JV (KPC)\nHossam Zidan\xa0\nApache Egypt JV (KPC)\nNihal Darraj\xa0\nDepartment of Earth Science and Engineering, Imperial College\nArticle\nKeywords:\nPosted Date: June 14th, 2024\nDOI: https://doi.org/10.21203/rs.3.rs-4517115/v1\nLicense: \uf25e \uf4e7 This work is licensed under a Creative Commons Attribution 4.0 International License. \xa0\nRead Full License\nAdditional Declarations: No competing interests reported.'),
 Document(metadata={'producer'

In [17]:
#Embedding and storing
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

db=Chroma.from_documents(documents,GoogleGenerativeAIEmbeddings(model='gemini-embedding-001'))




In [18]:
#Advanced Querying using Chain and Retriever
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
   """
Answer the following question based only on the provided context. 
Think step by step before providing a detailed answer. 
I will tip you $1000 if the user finds the answer helpful. 
<context>
{context}
</context>
Question: {input}""")



In [19]:
from langchain_community.llms import Ollama
## Load Ollama LAMA2 LLM model
llm=Ollama(model="llama2")
llm

C:\Users\Sanjana S Naik\AppData\Local\Temp\ipykernel_3112\2277366689.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm=Ollama(model="llama2")


Ollama()

In [ ]:
#Creating the chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

chain=create_stuff_documents_chain(llm,prompt)

In [ ]:
#Creating the retriever
retriver=db.as_retriever()

In [ ]:
#Retriever chain
from langchain_classic.chains import create_retrieval_chain
retriever_chain=create_retrieval_chain(retriver,chain)


In [ ]:
#Invoking the Retriever chain
response=retriever_chain.invoke({"input":"What is the abstract in the document"})

In [35]:
response['answer']

'The abstract of the document you provided is:\n\n"This research explores a new method for monitoring and diagnosing oil wells in artificial lift systems, particularly those using sucker rod pumps (SRP). By combining dynamometer cards, computer vision, and deep learning techniques, we aim to enhance traditional dynamometer card analysis. Our approach involves advanced computer vision and deep learning algorithms to automate interpretation and improve accuracy in identifying issues such as fluid pound, gas interference, and tubing leaks. The proposed method has the potential to minimize non-productive time, reduce costs associated with pump replacements and workover operations, and foster a collaborative environment for problem-solving and resource management. The integration of dynamometer cards, computer vision, and deep learning techniques offers a more accurate and efficient way to monitor and diagnose SRP systems, ultimately driving progress and efciency in field operations."'